# Taller 9: Objetos básico (continuación)

Continuación del Taller 5. Ahí construiste clases simples con atributos y métodos. Aquí subimos un peldaño hacia herramientas que hacen a los objetos más robustos y expresivos: **encapsulación** con atributos protegidos, **métodos especiales** (`__str__`, `__eq__`, `__lt__`) para que tus objetos se impriman y se comparen de forma natural, **herencia** para reutilizar y extender comportamiento, y **`classmethod`/`staticmethod`** para funcionalidad que pertenece a la clase pero no depende de una instancia particular.

> Enunciado, modelo de pensamiento y solución con type hints en cada ejercicio, más el análisis de complejidad temporal y espacial.

## Ejercicio 1: Encapsulación: proteger un atributo con validación

### Enunciado

Escribe una clase `CuentaBancaria` con un atributo `_saldo` (protegido por convención) que **no** pueda volverse negativo. Expón el saldo como una `property` de solo lectura, y agrega métodos `depositar(monto)` y `retirar(monto)` que validen que el monto sea positivo y que, en el caso de `retirar`, no dejen el saldo en negativo (lanzando `ValueError` si la operación no es válida).

### Modelo de pensamiento

1. En el Taller 5 los atributos eran públicos y cualquiera podía modificarlos directamente (`estudiante.nota = -50`, sin ningún control). El prefijo `_` antes de un nombre de atributo (`_saldo`) es una **convención** de Python: le dice a quien lea el código "esto es interno, no lo modifiques directamente desde afuera de la clase" — Python no lo impide técnicamente, pero es la señal aceptada de "trátalo como privado".
2. Para exponer `_saldo` de forma controlada, en vez de dejarlo totalmente oculto, se usa el decorador `@property`: convierte un método (`def saldo(self)`) en algo que se **lee** como si fuera un atributo (`cuenta.saldo`, sin paréntesis), pero cuyo valor en realidad viene de código que tú controlas — en este caso, simplemente devuelve `self._saldo`, pero podrías agregar lógica adicional sin cambiar cómo se usa desde afuera.
3. Toda la validación de reglas de negocio (montos positivos, no dejar saldo negativo) vive en los **métodos** (`depositar`, `retirar`), no en el atributo directamente — esa es la esencia de la encapsulación: el estado interno solo cambia a través de operaciones que garantizan que ese estado se mantenga válido.

In [1]:
class CuentaBancaria:
    def __init__(self, saldo_inicial: float = 0.0) -> None:
        if saldo_inicial < 0:
            raise ValueError("El saldo inicial no puede ser negativo")
        self._saldo = saldo_inicial

    @property
    def saldo(self) -> float:
        return self._saldo

    def depositar(self, monto: float) -> None:
        if monto <= 0:
            raise ValueError("El monto a depositar debe ser positivo")
        self._saldo += monto

    def retirar(self, monto: float) -> None:
        if monto <= 0:
            raise ValueError("El monto a retirar debe ser positivo")
        if monto > self._saldo:
            raise ValueError("Saldo insuficiente")
        self._saldo -= monto


cuenta = CuentaBancaria(100.0)
cuenta.depositar(50.0)
cuenta.retirar(30.0)
print(cuenta.saldo)  # 120.0

try:
    cuenta.retirar(1000.0)
except ValueError as error:
    print("Error esperado:", error)


120.0
Error esperado: Saldo insuficiente


**Complejidad:** cada operación (`depositar`, `retirar`, leer `saldo`) hace una cantidad fija de comparaciones y una suma o resta → **O(1) en tiempo y en espacio**, sin importar cuántas operaciones se hayan hecho antes: la clase solo guarda un número, no un historial.

## Ejercicio 2: `__str__` y `__repr__`: representación legible de un objeto

### Enunciado

Retoma la clase `Estudiante` del Taller 5 (con `nombre` y `notas: list[float]`) y agrégale un método `__str__` que produzca un texto legible como `"Ana (promedio: 4.20)"` cuando se use `print(estudiante)` o `str(estudiante)`.

### Modelo de pensamiento

1. Sin `__str__`, imprimir un objeto muestra algo poco útil como `<__main__.Estudiante object at 0x7f...>` — la dirección de memoria del objeto, no información sobre sus datos. `__str__` es un **método especial** (también llamado *dunder*, por los dobles guiones bajos `__`): Python lo llama automáticamente cada vez que necesita convertir el objeto a texto, sin que tengas que invocarlo tú mismo con ese nombre.
2. Dentro de `__str__` puedes reutilizar otros métodos de la propia clase (como un método `promedio()` que ya calcula el promedio de `self.notas`), en vez de recalcular la lógica ahí mismo — evitando duplicar el cálculo del promedio en dos lugares distintos.
3. Como buena práctica adicional (no pedida explícitamente, pero relacionada) también se puede definir `__repr__`, pensado para depuración (lo que se muestra en una consola interactiva o dentro de una lista de objetos), normalmente más detallado que `__str__`. Aquí se agregan ambos para dejar claro cuándo se usa cada uno.

In [2]:
class Estudiante:
    def __init__(self, nombre: str, notas: list[float]) -> None:
        self.nombre = nombre
        self.notas = notas

    def promedio(self) -> float:
        return sum(self.notas) / len(self.notas)

    def __str__(self) -> str:
        return f"{self.nombre} (promedio: {self.promedio():.2f})"

    def __repr__(self) -> str:
        return f"Estudiante(nombre={self.nombre!r}, notas={self.notas!r})"


ana = Estudiante("Ana", [4.5, 3.9, 4.2])
print(ana)         # usa __str__: Ana (promedio: 4.20)
print(repr(ana))    # usa __repr__: Estudiante(nombre='Ana', notas=[4.5, 3.9, 4.2])


Ana (promedio: 4.20)
Estudiante(nombre='Ana', notas=[4.5, 3.9, 4.2])


**Complejidad:** `__str__` llama a `promedio()`, que recorre la lista de notas una vez → **O(n) en tiempo**, con `n` el número de notas del estudiante. `__repr__` construye un string a partir de los atributos existentes sin recorrer nada adicional (`notas` se formatea completa, así que también depende de `n`) → **O(n)** igualmente. En **espacio**, ambos solo generan un string nuevo de tamaño proporcional a los datos mostrados → **O(n)**, sin estructuras adicionales persistentes.

## Ejercicio 3: `__eq__` y `__lt__`: comparar y ordenar objetos

### Enunciado

Agrega a `Estudiante` los métodos `__eq__` (dos estudiantes son iguales si tienen el mismo promedio) y `__lt__` (un estudiante es "menor" que otro si tiene menor promedio), de forma que se pueda usar `sorted()` directamente sobre una lista de objetos `Estudiante`, sin necesitar un `key=` explícito.

### Modelo de pensamiento

1. Por defecto, Python compara objetos por **identidad** (si son literalmente el mismo objeto en memoria), no por sus datos — por eso `Estudiante("Ana", [5.0]) == Estudiante("Ana", [5.0])` sería `False` sin ayuda. `__eq__` redefine qué significa `==` para tu clase: aquí, "igual" se define como "mismo promedio", una decisión de diseño explícita del enunciado (podrías haber elegido comparar por nombre, o por todos los atributos a la vez).
2. `__lt__` (*less than*, `<`) es lo que Python necesita internamente para poder ordenar objetos de tu clase: `sorted()` compara pares de elementos con `<` para decidir su orden relativo. Al definir `__lt__` una sola vez, obtienes gratis la posibilidad de usar `sorted()`, `min()` y `max()` directamente sobre listas de estudiantes.
3. Nota el patrón de la firma: `__eq__(self, otro: object) -> bool` — el parámetro se tipa como `object` (no como `Estudiante`) porque Python puede llamar a `__eq__` al comparar contra *cualquier* tipo (por ejemplo, `estudiante == 5`), y tu método debe poder manejar ese caso con seguridad, por eso se valida `isinstance(otro, Estudiante)` antes de acceder a `otro.promedio()`.

In [3]:
class Estudiante:
    def __init__(self, nombre: str, notas: list[float]) -> None:
        self.nombre = nombre
        self.notas = notas

    def promedio(self) -> float:
        return sum(self.notas) / len(self.notas)

    def __str__(self) -> str:
        return f"{self.nombre} (promedio: {self.promedio():.2f})"

    def __eq__(self, otro: object) -> bool:
        if not isinstance(otro, Estudiante):
            return NotImplemented
        return self.promedio() == otro.promedio()

    def __lt__(self, otro: "Estudiante") -> bool:
        return self.promedio() < otro.promedio()


estudiantes = [
    Estudiante("Ana", [4.2, 4.5]),
    Estudiante("Luis", [3.0, 3.5]),
    Estudiante("Marta", [4.9, 5.0]),
]

for estudiante in sorted(estudiantes):
    print(estudiante)
# Luis (promedio: 3.25)
# Ana (promedio: 4.35)
# Marta (promedio: 4.95)


Luis (promedio: 3.25)
Ana (promedio: 4.35)
Marta (promedio: 4.95)


**Complejidad:** cada comparación (`__eq__` o `__lt__`) calcula el promedio de ambos estudiantes, es decir, recorre sus listas de notas → **O(m) por comparación**, con `m` el número de notas de cada estudiante (asumiendo listas de tamaño similar). `sorted()` sobre `e` estudiantes hace **O(e log e)** comparaciones (Timsort), cada una de costo O(m) → **O(e × m × log e) en tiempo** en total. En **espacio**, `sorted()` construye una lista nueva de tamaño `e` → **O(e)** adicional (sin contar el espacio ya ocupado por los objetos).

## Ejercicio 4: Herencia: extender `Estudiante` con una beca

### Enunciado

Crea una clase `EstudianteBecado` que **herede** de `Estudiante` y agregue un atributo `porcentaje_beca` (por ejemplo, `50` para media beca). Sobrescribe `__str__` para que además muestre el porcentaje de beca, reutilizando la lógica de la clase base en vez de reescribirla desde cero.

### Modelo de pensamiento

1. La herencia (`class EstudianteBecado(Estudiante):`) expresa una relación **"es un"**: un `EstudianteBecado` *es un* `Estudiante`, con todo lo que eso implica (tiene `nombre`, `notas`, `promedio()`), más algo adicional. Es distinto de la **composición** que ya usaste en el Taller 5 (donde `Curso` *tenía una* lista de estudiantes, relación "tiene un") — aquí la relación es de tipo, no de contenido.
2. Dentro de `__init__` de la subclase, en vez de repetir `self.nombre = nombre` y `self.notas = notas`, se llama a `super().__init__(nombre, notas)`: eso delega la inicialización de los atributos heredados a la clase base, evitando duplicar esa lógica. Si `Estudiante.__init__` cambiara en el futuro, `EstudianteBecado` seguiría funcionando sin tocarla.
3. Para `__str__`, la técnica es la misma idea aplicada a un método: en vez de reescribir todo el formato de texto desde cero, se llama a `super().__str__()` para obtener la parte que ya sabe construir la clase base, y solo se le agrega la información nueva (el porcentaje de beca) al final. Esto es **sobrescritura con reutilización**, más robusto que copiar y pegar el formato completo en la subclase.

In [4]:
class Estudiante:
    def __init__(self, nombre: str, notas: list[float]) -> None:
        self.nombre = nombre
        self.notas = notas

    def promedio(self) -> float:
        return sum(self.notas) / len(self.notas)

    def __str__(self) -> str:
        return f"{self.nombre} (promedio: {self.promedio():.2f})"


class EstudianteBecado(Estudiante):
    def __init__(self, nombre: str, notas: list[float], porcentaje_beca: float) -> None:
        super().__init__(nombre, notas)
        self.porcentaje_beca = porcentaje_beca

    def __str__(self) -> str:
        return f"{super().__str__()} [beca {self.porcentaje_beca:.0f}%]"


becado = EstudianteBecado("Pedro", [4.0, 4.5, 4.2], porcentaje_beca=50)
print(becado)  # Pedro (promedio: 4.23) [beca 50%]
print(isinstance(becado, Estudiante))  # True: un EstudianteBecado también es un Estudiante


Pedro (promedio: 4.23) [beca 50%]
True


**Complejidad:** `super().__init__(...)` y `super().__str__()` hacen exactamente el mismo trabajo que ya se analizó para `Estudiante` (O(1) para inicializar, O(n) para `__str__` por el promedio, con `n` el número de notas); agregar el sufijo de la beca es O(1) adicional. En total, la subclase mantiene la misma complejidad que la clase base: **O(n) en tiempo** para `__str__`, **O(1)** para inicializar. En **espacio**, `EstudianteBecado` agrega un único atributo numérico (`porcentaje_beca`) sobre lo que ya ocupaba `Estudiante` → **O(1)** adicional.

## Ejercicio 5: `classmethod` y `staticmethod`

### Enunciado

Agrega a `Estudiante` dos formas alternativas de trabajar con la clase, sin necesitar una instancia ya creada: un **classmethod** `desde_texto(cls, texto)` que construya un `Estudiante` a partir de un string con formato `"nombre:nota1,nota2,nota3"`, y un **staticmethod** `es_nota_valida(nota)` que valide si un número está en el rango `[0.0, 5.0]`, sin depender de ningún estudiante en particular.

### Modelo de pensamiento

1. Un método normal (con `self`) necesita una instancia ya construida para poder usarse (`estudiante.promedio()`). Pero a veces la operación que quieres tiene sentido **antes** de que exista una instancia — como "construir un estudiante a partir de otra representación de datos" (aquí, un string). Para eso sirve `@classmethod`: en vez de `self`, recibe `cls` (la clase misma), y típicamente termina construyendo y devolviendo una instancia nueva (`return cls(...)`), funcionando como un "constructor alternativo" con nombre propio y más descriptivo que `__init__`.
2. `desde_texto` necesita **parsear** el string: separar nombre y notas por `:`, y luego las notas entre sí por `,`, convirtiendo cada una a `float`. Es el mismo patrón de "dividir un problema en pasos" — primero separar, luego convertir tipo por tipo.
3. `es_nota_valida`, en cambio, no necesita ni una instancia (`self`) ni la clase (`cls`) — es una función que simplemente *vive* dentro de la clase porque conceptualmente pertenece ahí (es una regla sobre qué es una "nota válida" para esta clase), pero no usa ni depende de ningún dato de una instancia particular. Para ese caso se usa `@staticmethod`, que no recibe ni `self` ni `cls` — es básicamente una función normal agrupada dentro del espacio de nombres de la clase, por claridad de organización.

In [5]:
class Estudiante:
    def __init__(self, nombre: str, notas: list[float]) -> None:
        self.nombre = nombre
        self.notas = notas

    def promedio(self) -> float:
        return sum(self.notas) / len(self.notas)

    @staticmethod
    def es_nota_valida(nota: float) -> bool:
        return 0.0 <= nota <= 5.0

    @classmethod
    def desde_texto(cls, texto: str) -> "Estudiante":
        nombre, notas_texto = texto.split(":")
        notas = [float(nota) for nota in notas_texto.split(",")]
        for nota in notas:
            if not cls.es_nota_valida(nota):
                raise ValueError(f"Nota inválida: {nota}")
        return cls(nombre, notas)


ana = Estudiante.desde_texto("Ana:4.5,3.9,4.2")
print(ana.nombre, ana.notas, ana.promedio())
print(Estudiante.es_nota_valida(4.5))   # True
print(Estudiante.es_nota_valida(7.0))   # False


Ana [4.5, 3.9, 4.2] 4.2
True
False


**Complejidad:** `es_nota_valida` hace dos comparaciones fijas → **O(1)**. `desde_texto` divide el texto en `m` notas y valida cada una con `es_nota_valida` (O(1) cada vez) → **O(m) en tiempo**, con `m` el número de notas en el texto. En **espacio**, se construye una lista de `m` notas más la instancia resultante → **O(m)**.

## Ejercicio 6: Composición de tres niveles: `Universidad` → `Curso` → `Estudiante`

### Enunciado

Retoma la clase `Curso` del Taller 5 (contiene una lista de `Estudiante`) y crea una clase `Universidad` que contenga una lista de `Curso`. Agrega un método `mejor_estudiante_general()` que devuelva el `Estudiante` con mayor promedio **entre todos los cursos** de la universidad (no el mejor de cada curso por separado).

### Modelo de pensamiento

1. Esta es composición de tres niveles: `Universidad` tiene `Curso`s, y cada `Curso` tiene `Estudiante`s. Para llegar del nivel más alto (`Universidad`) al más bajo (cada `Estudiante`) hay que atravesar **dos** niveles de anidamiento — un ciclo dentro de otro, uno por cada nivel de contención, muy parecido en estructura al ciclo anidado para recorrer una matriz (Ejercicio 3 del Taller 7), solo que aquí "las filas" son cursos y "las columnas" son estudiantes dentro de cada curso.
2. En vez de escribir ese doble ciclo dentro de `mejor_estudiante_general`, conviene apoyarse en que `Curso` ya sabería encontrar a su propio mejor estudiante (reutilizando lo que hiciste en el Taller 5) — así, el ciclo externo (sobre los cursos) solo necesita comparar **un candidato por curso**, no revisar estudiante por estudiante directamente. Esto es el mismo principio de descomposición del Taller 6, aplicado a clases: cada nivel resuelve su propia parte del problema, y el nivel superior combina esos resultados.
3. El caso borde a considerar: ¿qué pasa si algún curso no tiene estudiantes, o la universidad no tiene cursos? Conviene decidir explícitamente qué hacer (por ejemplo, ignorar cursos vacíos, o lanzar un error si no hay ningún estudiante en toda la universidad) en vez de dejar que el programa falle con un error críptico.

In [6]:
class Estudiante:
    def __init__(self, nombre: str, notas: list[float]) -> None:
        self.nombre = nombre
        self.notas = notas

    def promedio(self) -> float:
        return sum(self.notas) / len(self.notas)


class Curso:
    def __init__(self, nombre: str, estudiantes: list[Estudiante]) -> None:
        self.nombre = nombre
        self.estudiantes = estudiantes

    def mejor_estudiante(self) -> Estudiante | None:
        if not self.estudiantes:
            return None
        mejor = self.estudiantes[0]
        for estudiante in self.estudiantes[1:]:
            if estudiante.promedio() > mejor.promedio():
                mejor = estudiante
        return mejor


class Universidad:
    def __init__(self, cursos: list[Curso]) -> None:
        self.cursos = cursos

    def mejor_estudiante_general(self) -> Estudiante | None:
        mejor_general: Estudiante | None = None
        for curso in self.cursos:
            candidato = curso.mejor_estudiante()
            if candidato is None:
                continue
            if mejor_general is None or candidato.promedio() > mejor_general.promedio():
                mejor_general = candidato
        return mejor_general


programacion = Curso("Programación", [Estudiante("Ana", [4.5, 4.0]), Estudiante("Luis", [3.0, 3.5])])
estructuras = Curso("Estructuras", [Estudiante("Marta", [4.9, 5.0]), Estudiante("Pedro", [3.8, 4.0])])

udem = Universidad([programacion, estructuras])
mejor = udem.mejor_estudiante_general()
print(mejor.nombre if mejor else "Sin estudiantes")  # Marta


Marta


**Complejidad:** sea `k` el número de cursos y `e` el número (promedio) de estudiantes por curso, y `m` el número de notas por estudiante. `Curso.mejor_estudiante()` recorre sus `e` estudiantes, calculando un promedio de O(m) cada vez → **O(e × m)** por curso. `Universidad.mejor_estudiante_general()` llama a eso una vez por cada uno de los `k` cursos → **O(k × e × m) en tiempo** en total — equivalente a recorrer una vez a cada estudiante de toda la universidad y calcular su promedio, sin trabajo redundante. En **espacio**, no se crean estructuras nuevas proporcionales a los datos (solo se guardan referencias a un "mejor" candidato a la vez) → **O(1)** adicional, más allá de los objetos que ya existían.

## Ejercicio 7: Sistema de biblioteca con herencia y polimorfismo (reto)

### Enunciado

Diseña una clase base `MaterialBibliotecario` con `titulo` y un método `descripcion()` que devuelva un string. Crea dos subclases: `Libro` (agrega `autor` y `paginas`) y `Revista` (agrega `numero_edicion`), cada una sobrescribiendo `descripcion()` con su propio formato. Luego escribe una función `imprimir_catalogo(materiales)` que reciba una lista mezclada de libros y revistas, y llame a `descripcion()` sobre cada uno **sin preguntar de qué tipo es cada elemento** (sin usar `isinstance` ni `if`).

### Modelo de pensamiento

1. El punto central de este ejercicio es el **polimorfismo**: distintas subclases pueden responder de forma distinta al mismo mensaje (`descripcion()`), y quien llama al método no necesita saber ni preguntar qué subclase específica está manejando — solo confía en que, sea cual sea el tipo concreto, todos entienden `descripcion()` porque todos heredan de `MaterialBibliotecario`.
2. Esto se logra con **sobrescritura de métodos**: la clase base define `descripcion()` con una implementación por defecto (o la deja para que cada subclase la complete), y cada subclase la reemplaza con su propia versión. Cuando llamas `material.descripcion()` sobre un elemento de una lista mixta, Python decide automáticamente **en tiempo de ejecución** cuál versión del método ejecutar, según el tipo real del objeto — a esto se le llama *despacho dinámico*, y es lo que hace innecesario el `if isinstance(...)` que pide evitar el enunciado.
3. `imprimir_catalogo` entonces se vuelve una función muy simple: un ciclo que llama `.descripcion()` sobre cada elemento, sin ninguna lógica condicional sobre el tipo — toda la "inteligencia" sobre cómo describirse vive dentro de cada clase, no en quien las usa. Esa separación (cada clase sabe describirse a sí misma) es la idea de diseño más importante del ejercicio, más allá del código en sí.

In [7]:
class MaterialBibliotecario:
    def __init__(self, titulo: str) -> None:
        self.titulo = titulo

    def descripcion(self) -> str:
        return f"{self.titulo}"


class Libro(MaterialBibliotecario):
    def __init__(self, titulo: str, autor: str, paginas: int) -> None:
        super().__init__(titulo)
        self.autor = autor
        self.paginas = paginas

    def descripcion(self) -> str:
        return f"Libro: '{self.titulo}' de {self.autor} ({self.paginas} páginas)"


class Revista(MaterialBibliotecario):
    def __init__(self, titulo: str, numero_edicion: int) -> None:
        super().__init__(titulo)
        self.numero_edicion = numero_edicion

    def descripcion(self) -> str:
        return f"Revista: '{self.titulo}', edición #{self.numero_edicion}"


def imprimir_catalogo(materiales: list[MaterialBibliotecario]) -> None:
    for material in materiales:
        print(material.descripcion())


catalogo = [
    Libro("Cien años de soledad", "García Márquez", 471),
    Revista("National Geographic", 245),
    Libro("El principito", "Saint-Exupéry", 96),
]

imprimir_catalogo(catalogo)


Libro: 'Cien años de soledad' de García Márquez (471 páginas)
Revista: 'National Geographic', edición #245
Libro: 'El principito' de Saint-Exupéry (96 páginas)


**Complejidad:** `imprimir_catalogo` recorre la lista una vez, y cada llamada a `descripcion()` hace un trabajo O(1) (construir un string con los atributos ya guardados, sin recorrer estructuras adicionales) → **O(n) en tiempo**, con `n` el número de materiales en el catálogo. En **espacio**, no se construyen estructuras nuevas proporcionales a `n` (solo se imprime cada descripción) → **O(1) adicional** más allá de la lista de entrada; si en vez de imprimir quisieras **acumular** las descripciones en una lista de strings, el espacio adicional pasaría a ser O(n).